
- Document data context and data sampling in markdown

# Data Context: Welltory COVID-19 & Wearables Open Research

## Context & Background
- **Source:** Welltory Open Research Initiative (`Welltory/hrv-covid19`).
- **Objective:** Track heart rate variability (HRV), vital signs, and self-reported symptoms of individuals with positive or suspected COVID-19 statuses to identify predictive digital biomarkers for disease progression and recovery.
- **Ethical & Privacy Considerations:** Data is fully anonymized. It contains no personally identifiable information (PII) such as names or emails.

## Sampling Methodology & Scope
- **Population:** 186 international participants representing various stages of COVID-19 infection (pre-symptomatic, acute, and recovery phases).
- **Collection Mechanism:** 
  1. *Optical PPG:* Smartphone camera-based pulse wave recordings.
  2. *Wearables:* Smart devices (Apple Watch, Garmin, Fitbit) syncing via Apple Health or Google Fit.
  3. *Surveys:* Clinically structured self-assessments detailing symptom severities and testing timelines.
- **Sampling Bias Note:** Convenience sampling of tech-literate users who actively use a consumer health application (Welltory). Selection bias toward individuals healthy enough or concerned enough to track metrics daily.

In [1]:
# data exploration and cleaning

import pandas as pd
import numpy as np

DATA = "data/"

participants = pd.read_csv(DATA + "participants.csv")
hrv = pd.read_csv(DATA + "hrv_measurements.csv", parse_dates=["measurement_datetime"])
heart_rate = pd.read_csv(DATA + "heart_rate.csv", parse_dates=["datetime"])
blood_pres = pd.read_csv(DATA + "blood_pressure.csv", parse_dates=["measurement_datetime"])
sleep = pd.read_csv(DATA + "sleep.csv", parse_dates=["day", "sleep_begin", "sleep_end"])
wearables = pd.read_csv(DATA + "wearables.csv", parse_dates=["day"])
weather = pd.read_csv(DATA + "weather.csv", parse_dates=["day"])
surveys = pd.read_csv(DATA + "surveys.csv", parse_dates=["created_at"])
scales = pd.read_csv(DATA + "scales_description.csv")

dfs = {
    "participants": participants,
    "hrv": hrv,
    "heart_rate": heart_rate,
    "blood_pressure": blood_pres,
    "sleep": sleep,
    "wearables": wearables,
    "weather": weather,
    "surveys": surveys,
    "scales": scales,
}

summary = pd.DataFrame({
    name: {
        "rows": len(df),
        "columns": df.shape[1],
        "unique users": df["user_code"].nunique() if "user_code" in df else np.nan,
        "duplicate rows": int(df.duplicated().sum()),
        "% missing": round(df.isna().mean().mean() * 100, 1),
    }
    for name, df in dfs.items()
}).T
print(summary, "\n")


def top_corr(df, n=5, min_periods=30):
    num = df.select_dtypes("number")
    num = num.loc[:, num.nunique() > 1]
    c = num.corr(min_periods=min_periods)
    pairs = c.where(np.triu(np.ones(c.shape, dtype=bool), k=1)).stack()
    return pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(n).round(2)


for name in ["participants", "hrv", "blood_pressure", "sleep", "wearables", "weather"]:
    print(f"top correlations: {name}")
    print(top_corr(dfs[name]).to_string(), "\n")

user_hrv = hrv.groupby("user_code")[["bpm", "rmssd", "sdnn", "how_feel", "how_mood"]].mean().add_prefix("hrv_")
user_bp = blood_pres.groupby("user_code")[["systolic", "diastolic"]].mean()
user_wear = wearables.groupby("user_code")[["resting_pulse", "steps_count"]].mean()

per_user = participants.set_index("user_code")[["height", "weight"]].join([user_hrv, user_bp, user_wear])
per_user["bmi"] = per_user["weight"] / (per_user["height"] / 100) ** 2
print("top correlations across tables (per user)")
print(top_corr(per_user, n=10, min_periods=20).to_string())

                    rows  columns  unique users  duplicate rows  % missing
participants       185.0      8.0         185.0             0.0        3.9
hrv               3245.0     22.0         185.0             0.0        4.0
heart_rate      523783.0      4.0          79.0             0.0        0.0
blood_pressure     721.0      8.0          28.0             0.0       29.5
sleep              425.0     12.0          10.0             0.0       56.3
wearables         3098.0     18.0          79.0             0.0       48.2
weather           1717.0      7.0         104.0             0.0        0.0
surveys           2259.0      5.0         111.0             0.0        0.0
scales             148.0      4.0           NaN             0.0        0.0 

top correlations: participants
height  weight    0.23 

top correlations: hrv
meanrr  mode           0.98
bpm     meanrr        -0.98
        mode          -0.96
mxdmn   sdnn           0.95
hf      total_power    0.91 

top correlations: blood_pres

In [ ]:
#- Explore data visually with appropriate visualizations

import matplotlib.pyplot as plt

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e1e0d9"
POSITIVE, NEGATIVE = "#2a78d6", "#e34948"

covid_wide = (
    surveys[surveys["scale"].str.startswith(("S_COVID", "S_CORONA"))]
    .pivot_table(index="user_code", columns="scale", values="value", aggfunc="mean")
)
covid_wide = covid_wide.loc[:, covid_wide.notna().sum() >= 20]
user_hrv_mean = hrv.groupby("user_code")[["rmssd", "bpm"]].mean()
joined = covid_wide.join(user_hrv_mean, how="inner")

labels = {
    "S_CORONA": "Coronavirus symptoms",
    "S_COVID_OVERALL": "Overall feeling",
    "S_COVID_BLUISH": "Bluish skin",
    "S_COVID_BREATH": "Shortness of breath",
    "S_COVID_CONFUSION": "Confusion",
    "S_COVID_COUGH": "Cough",
    "S_COVID_FATIGUE": "Fatigue",
    "S_COVID_FEVER": "Fever",
    "S_COVID_PAIN": "Pain",
    "S_COVID_TROUBLE": "Trouble (other)",
}
scale_cols = [c for c in covid_wide.columns if c in labels]
corr = joined[scale_cols + ["rmssd", "bpm"]].corr().loc[scale_cols, ["rmssd", "bpm"]]
corr = corr.sort_values("rmssd")

panels = [
    ("rmssd", "Correlation with HRV (RMSSD)", "higher = more HRV"),
    ("bpm", "Correlation with heart rate (bpm)", "higher = faster pulse"),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True, facecolor=SURFACE)
y = range(len(corr))

for ax, (col, title, note) in zip(axes, panels):
    vals = corr[col]
    ax.set_facecolor(SURFACE)
    ax.barh(list(y), vals, height=0.55, color=[POSITIVE if v >= 0 else NEGATIVE for v in vals])
    ax.axvline(0, color=MUTED, linewidth=1)
    ax.set_xlim(-0.5, 0.5)
    ax.set_yticks(list(y))
    ax.set_yticklabels([labels[c] for c in corr.index], color=INK)
    ax.tick_params(axis="x", colors=MUTED, length=0)
    ax.tick_params(axis="y", length=0)
    ax.xaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.set_title(title, loc="left", color=INK, fontsize=12, fontweight="bold")
    ax.set_xlabel(note, color=MUTED, fontsize=9)
    for yi, v in zip(y, vals):
        ax.text(v + (0.015 if v >= 0 else -0.015), yi, f"{v:+.2f}", va="center",
                ha="left" if v >= 0 else "right", color=INK, fontsize=9)

fig.suptitle(
    f"COVID symptom reports vs. HRV and heart rate (per user, n={len(joined)})",
    x=0.02, ha="left", color=INK, fontsize=13, fontweight="bold",
)
fig.text(0.02, 0.005, "Blue = positive correlation, red = negative. Survey values are per-user means.",
         color=MUTED, fontsize=9)
plt.tight_layout(rect=(0, 0.03, 1, 0.94))
plt.show()


Note the data above was selected so we could do a feature analysis test on how COVID impacted users. The graph is helpful as it provides clear way to track COVID features and how they link to one another: 

Some key features include: 

- how people feel when they have COVID based on symptoms 
- when COVID is onset how body tempreture heat might rise (note there could be a correlation with weather and humidity here)

#- Discuss and implement strategies for Handling Missing Values, Removing Duplicates, and Handling Outliers


And Outlliers will be marked as such and evalulated for validitiy - why was it an outlier - was it hardware based or a glitch in the collection of data. It will be removed from the data set once validated. 

Missing values will be removed unless a reasonable forecasting model can be created based on other feature. 

Duplicates will be removed so data is not double counted - this in our function analyzing duplicate data.

## COVID-19 and Heart Rate Variability

Question: do self-reported COVID-19 symptoms, or time since symptom onset, relate to HRV?

- **HRV measures:** `rmssd`, `sdnn`, `pnn50`, `total_power` (higher generally means better autonomic recovery), `lfhf` (sympathetic/parasympathetic balance) and `bpm`.
- **Same-day link:** COVID survey answers (`S_COVID_*`, `S_CORONA`) matched to the HRV recorded by the same user on the same day.
- **Symptom score:** the mean of the individual symptom scales (cough, fever, breath, fatigue, pain, confusion, trouble, bluish).
- **Onset link:** HRV before, during the first 2 weeks after, and after `symptoms_onset` for participants who report a date.

In [ ]:
#- Perform data transformation as appropriate (NA)

In [ ]:
#- Create at least one new feature and document your approach

- Include a discussion around data quality assessment, including data profiling, data completeness, data accuracy, data consistency, data integrity, and data lineage and provence

This data was collected from participants who wore the fitbits/ health care equipment and took surveys which means there are likely implict and explict bias. Survey results vary based on feeling of participants, additionally the particapants wearing the watch vs not wearing the watch - likely had different results as every person is different. With no clear experimental trails for more than half the data the quality of the data does not account for many factors - with many independent variables. Additionally, the data profiling is only for those who track there general fitness, which means it does not include those with no means to purchase a fit bit or those who struggle with consistency or is not fit. Data accuracy is based on the model and software of the watch - in the future I would want to narrow the scope of data analysis for just the apple watch with SW vesions - (---) this would allow for better variable management. Data consistency is also variable based on participant, and data provence from is from open source data which could or could not be accurate. As for data lineage as we custom built our feature we pulled from static data so there was no ingestion pipeline.
 